### Dataset

Splitting the dataset in training, validation and test keeping all the training pairs containing the same images in the same split to avoid data leakage. The split is done by trying to get a training/validation proportion which is roughly 80/20

In [ ]:

from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_DIR = Path("/kaggle/input/competitions/aml-challenge-2026-27")  # Change to your dataset path
VAL_RATIO = 0.20
SEED = 42

# ============================================================
# 2. LOAD DATA
# ============================================================

train_all = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

with open(DATA_DIR / "train_pairs.json") as f:
    train_pairs = {
        p["pair_id"]: p for p in json.load(f)["pairs"]
    }

with open(DATA_DIR / "test_pairs.json") as f:
    test_pairs = {
        p["pair_id"]: p for p in json.load(f)["pairs"]
    }

# Parse accepted target coordinates (possibly multiple per query)
train_all["target_points"] = train_all["target_points"].apply(
    json.loads
)

# Build full image paths without loading images into memory
for df in (train_all, test_df):
    df["source_path"] = df["source_image"].apply(
        lambda p: DATA_DIR / p
    )
    df["target_path"] = df["target_image"].apply(
        lambda p: DATA_DIR / p
    )

# ============================================================
# 3. GROUP CONNECTED IMAGES
# ============================================================
# Images appearing in the same pair must belong to the same
# split. Connected components also handle images reused in
# different pairs and in different source/target roles.

parent = {}

def find(x):
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    parent[find(a)] = find(b)

for src, tgt in zip(
    train_all["source_image"], train_all["target_image"]
):
    union(src, tgt)

train_all["group"] = train_all["source_image"].apply(find)

# ============================================================
# 4. TRAIN / VALIDATION SPLIT
# ============================================================
# GroupShuffleSplit prevents connected images from appearing
# in both training and validation.
# Select the split closest to the requested query ratio.

splitter = GroupShuffleSplit(
    n_splits=100,
    test_size=VAL_RATIO,
    random_state=SEED
)

splits = splitter.split(
    train_all,
    groups=train_all["group"]
)

train_idx, val_idx = min(
    splits,
    key=lambda split: abs(
        len(split[1]) / len(train_all) - VAL_RATIO
    )
)

train_df = train_all.iloc[train_idx].reset_index(drop=True)
val_df = train_all.iloc[val_idx].reset_index(drop=True)

# ============================================================
# 5. CHECK SPLIT INTEGRITY
# ============================================================

def image_set(df):
    return set(df["source_image"]) | set(df["target_image"])

assert image_set(train_df).isdisjoint(image_set(val_df))
assert image_set(train_all).isdisjoint(image_set(test_df))

assert set(train_df["pair_id"]).isdisjoint(
    set(val_df["pair_id"])
)

# ============================================================
# 6. SUMMARY
# ============================================================

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        f"{name:12s} | "
        f"Queries: {len(df):4d} | "
        f"Pairs: {df['pair_id'].nunique():3d} | "
        f"Images: {len(image_set(df)):3d}"
    )

print("\nValidation fraction:",
      round(len(val_df) / len(train_all), 3))

print("\nExample training query:")
display(train_df.head(1))


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


# ============================================================
# 1. CONFIGURATION
# ============================================================

IMAGE_SIZE = 518
BATCH_SIZE = 4

# ImageNet normalization used for DINOv2
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


# ============================================================
# 2. IMAGE PREPROCESSING
# ============================================================

class ImagePreprocessor:
    def __init__(self, size=518, mean=MEAN, std=STD):
        self.size = (size, size) if isinstance(size, int) else tuple(size)
        self.mean = mean
        self.std = std

    def __call__(self, image):
        original_w, original_h = image.size
        new_h, new_w = self.size

        image = TF.resize(
            image,
            [new_h, new_w],
            interpolation=InterpolationMode.BICUBIC,
            antialias=True
        )

        image = TF.to_tensor(image)
        image = TF.normalize(image, self.mean, self.std)

        # Conversion factors from original to resized coordinates
        scale = torch.tensor([
            new_w / original_w,
            new_h / original_h
        ], dtype=torch.float32)

        return image, scale


# ============================================================
# 3. PYTORCH DATASET
# ============================================================

class CorrespondenceDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load source and target images
        with Image.open(row["source_path"]) as img:
            source = img.convert("RGB")

        with Image.open(row["target_path"]) as img:
            target = img.convert("RGB")

        # Original dimensions [width, height]
        source_size = torch.tensor(source.size, dtype=torch.float32)
        target_size = torch.tensor(target.size, dtype=torch.float32)

        # Source query point in original coordinates
        source_xy = torch.tensor(
            [row["source_x"], row["source_y"]],
            dtype=torch.float32
        )

        # Apply preprocessing
        if self.transform is not None:
            source, source_scale = self.transform(source)
            target, target_scale = self.transform(target)

            source_xy_model = source_xy * source_scale
        else:
            source = TF.to_tensor(source)
            target = TF.to_tensor(target)

            source_scale = torch.ones(2)
            target_scale = torch.ones(2)
            source_xy_model = source_xy.clone()

        # Ground truth is available only for training/validation
        has_gt = "target_points" in self.df.columns

        target_points = (
            torch.tensor(
                row["target_points"], dtype=torch.float32
            ).reshape(-1, 2)
            if has_gt else None
        )

        return {
            "id": row["id"],
            "pair_id": row["pair_id"],
            "query_index": int(row["query_index"]),

            "source_image": source,
            "target_image": target,

            "source_xy": source_xy,
            "source_xy_model": source_xy_model,

            "source_size": source_size,
            "target_size": target_size,

            "source_scale": source_scale,
            "target_scale": target_scale,

            "target_points": target_points,
            "scale": float(row["scale"]) if has_gt else None
        }


# ============================================================
# 4. CUSTOM COLLATE FUNCTION
# ============================================================
# Each query can have a different number of accepted
# target points. Keep these annotations as a list.

def correspondence_collate(batch):

    stack_keys = [
        "source_image", "target_image",
        "source_xy", "source_xy_model",
        "source_size", "target_size",
        "source_scale", "target_scale"
    ]

    result = {
        key: torch.stack([item[key] for item in batch])
        for key in stack_keys
    }

    result.update({
        "id": [item["id"] for item in batch],
        "pair_id": [item["pair_id"] for item in batch],
        "query_index": torch.tensor([
            item["query_index"] for item in batch
        ]),
        "target_points": [
            item["target_points"] for item in batch
        ],
        "scale": [item["scale"] for item in batch]
    })

    return result


# ============================================================
# 5. CREATE DATASETS AND DATALOADERS
# ============================================================

preprocessor = ImagePreprocessor(size=IMAGE_SIZE)

train_dataset = CorrespondenceDataset(train_df, preprocessor)
val_dataset = CorrespondenceDataset(val_df, preprocessor)
test_dataset = CorrespondenceDataset(test_df, preprocessor)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "collate_fn": correspondence_collate,
    "pin_memory": torch.cuda.is_available()
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs
)


# ============================================================
# 6. CHECK OUTPUT
# ============================================================

batch = next(iter(val_loader))

print("Source images:", batch["source_image"].shape)
print("Target images:", batch["target_image"].shape)
print("Source points:", batch["source_xy_model"].shape)

print("\nExample query:")
print("ID:", batch["id"][0])
print("Original source point:", batch["source_xy"][0].tolist())
print("Resized source point:", batch["source_xy_model"][0].tolist())
print("Accepted targets:", batch["target_points"][0].tolist())
print("PCK scale:", batch["scale"][0])


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "dinov2_vits14"
FEATURE_LAYER = None       # None = final layer
NORMALIZE_FEATURES = True
FREEZE_BACKBONE = True

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# 2. DINOv2 FEATURE EXTRACTOR
# ============================================================

class DINOv2FeatureExtractor(nn.Module):

    def __init__(
        self,
        model_name="dinov2_vits14",
        feature_layer=None,
        normalize=True,
        freeze=True
    ):
        super().__init__()

        self.backbone = torch.hub.load(
            "facebookresearch/dinov2",
            model_name
        )

        self.patch_size = self.backbone.patch_size
        self.embed_dim = self.backbone.embed_dim
        self.feature_layer = feature_layer
        self.normalize = normalize

        if freeze:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.backbone.eval()

    def forward(self, images):

        B, C, H, W = images.shape

        assert H % self.patch_size == 0
        assert W % self.patch_size == 0

        # Extract spatial features
        if self.feature_layer is None:

            output = self.backbone.forward_features(images)
            tokens = output["x_norm_patchtokens"]

            # [B, N_patches, embed_dim]
            h = H // self.patch_size
            w = W // self.patch_size

            features = tokens.reshape(
                B, h, w, self.embed_dim
            ).permute(0, 3, 1, 2).contiguous()

        else:
            features = self.backbone.get_intermediate_layers(
                images,
                n=[self.feature_layer],
                reshape=True,
                norm=True
            )[0]

        # L2 normalization along feature dimension
        if self.normalize:
            features = F.normalize(
                features, p=2, dim=1
            )

        return features


# ============================================================
# 3. INITIALIZE MODEL
# ============================================================

feature_extractor = DINOv2FeatureExtractor(
    model_name=MODEL_NAME,
    feature_layer=FEATURE_LAYER,
    normalize=NORMALIZE_FEATURES,
    freeze=FREEZE_BACKBONE
).to(device)

feature_extractor.eval()


# ============================================================
# 4. TEST ON ONE BATCH
# ============================================================

batch = next(iter(val_loader))

source_images = batch["source_image"].to(device)
target_images = batch["target_image"].to(device)

with torch.inference_mode():

    source_features = feature_extractor(source_images)
    target_features = feature_extractor(target_images)


print("Source images:", source_images.shape)
print("Target images:", target_images.shape)

print("Source features:", source_features.shape)
print("Target features:", target_features.shape)

print("Patch size:", feature_extractor.patch_size)
print("Embedding dimension:", feature_extractor.embed_dim)


In [ ]:

import numpy as np
import pandas as pd
import json
import torch
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

SOURCE_SAMPLING = "nearest"  # "nearest" or "bilinear"
PCK_THRESHOLD = 0.10


# ============================================================
# 2. EXTRACT SOURCE KEYPOINT FEATURES
# ============================================================

def sample_source_features(
    features,
    source_xy,
    patch_size,
    method="nearest"
):
    """
    features:  [B, C, Hf, Wf]
    source_xy: [B, 2] in resized image coordinates

    Returns:
        descriptors: [B, C]
    """

    B, C, Hf, Wf = features.shape

    if method == "nearest":

        # Convert image coordinates to patch indices
        x = torch.floor(
            source_xy[:, 0] / patch_size
        ).long().clamp(0, Wf - 1)

        y = torch.floor(
            source_xy[:, 1] / patch_size
        ).long().clamp(0, Hf - 1)

        # Flatten spatial dimensions
        flat_features = features.flatten(2)  # [B,C,Hf*Wf]

        indices = y * Wf + x

        descriptors = flat_features.gather(
            dim=2,
            index=indices[:, None, None].expand(-1, C, 1)
        ).squeeze(-1)

    elif method == "bilinear":

        # Coordinates normalized to [-1, 1]
        # align_corners=False: patch centers are respected
        grid = source_xy / source_xy.new_tensor([
            Wf * patch_size,
            Hf * patch_size
        ])

        grid = 2.0 * grid - 1.0
        grid = grid.view(B, 1, 1, 2)

        descriptors = F.grid_sample(
            features,
            grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=False
        )[:, :, 0, 0]

    else:
        raise ValueError(f"Unknown sampling method: {method}")

    return F.normalize(descriptors, dim=1)


# ============================================================
# 3. COSINE SIMILARITY MATCHING
# ============================================================

def match_features(
    source_features,
    target_features,
    source_xy,
    target_scale,
    target_size,
    patch_size,
    sampling="nearest"
):
    """
    Returns:
        predictions: [B, 2] in original target coordinates
        confidence:  [B] maximum cosine similarity
    """

    B, C, Ht, Wt = target_features.shape

    # Source descriptor at the query location
    query = sample_source_features(
        source_features,
        source_xy,
        patch_size,
        method=sampling
    )  # [B, C]

    # Target descriptors
    target_flat = target_features.flatten(2)
    target_flat = F.normalize(target_flat, dim=1)

    # Cosine similarity against every target patch
    similarity = torch.bmm(
        query.unsqueeze(1), target_flat
    ).squeeze(1)  # [B, Ht*Wt]

    confidence, best_idx = similarity.max(dim=1)

    # Flattened index -> grid coordinates
    x_patch = best_idx % Wt
    y_patch = torch.div(
        best_idx, Wt, rounding_mode="floor"
    )

    # Grid coordinates -> resized image coordinates
    # Use the center of the selected patch
    prediction_model = torch.stack(
        [x_patch, y_patch], dim=1
    ).float()

    prediction_model = (
        prediction_model + 0.5
    ) * patch_size

    # Resized coordinates -> original target coordinates
    predictions = prediction_model / target_scale

    # Ensure predictions stay inside the original image
    predictions = torch.minimum(
        predictions.clamp(min=0),
        target_size - 1
    )

    return predictions, confidence


# ============================================================
# 4. PREDICTION ON A DATALOADER
# ============================================================

@torch.inference_mode()
def predict_loader(loader, model, sampling="nearest"):

    model.eval()
    model_device = next(model.parameters()).device
    records = []

    for batch in loader:

        source_images = batch["source_image"].to(model_device)
        target_images = batch["target_image"].to(model_device)

        source_features = model(source_images)
        target_features = model(target_images)

        predictions, confidence = match_features(
            source_features=source_features,
            target_features=target_features,
            source_xy=batch["source_xy_model"].to(model_device),
            target_scale=batch["target_scale"].to(model_device),
            target_size=batch["target_size"].to(model_device),
            patch_size=model.patch_size,
            sampling=sampling
        )

        predictions = predictions.cpu().numpy()
        confidence = confidence.cpu().numpy()

        for i, query_id in enumerate(batch["id"]):
            records.append({
                "id": query_id,
                "x": float(predictions[i, 0]),
                "y": float(predictions[i, 1]),
                "confidence": float(confidence[i])
            })

    return pd.DataFrame(records)


# ============================================================
# 5. PCK EVALUATION
# ============================================================

def evaluate_pck(predictions, ground_truth, alpha=0.10):
    """
    PCK with multiple accepted target points.

    A prediction is correct if:
        min_distance / scale <= alpha
    """

    df = predictions.merge(
        ground_truth[["id", "target_points", "scale"]],
        on="id",
        how="inner",
        validate="one_to_one"
    )

    assert len(df) == len(ground_truth)

    normalized_errors = []

    for row in df.itertuples():

        pred = np.array([row.x, row.y], dtype=float)
        targets = np.asarray(row.target_points, dtype=float)

        # Minimum distance to any accepted target
        distances = np.linalg.norm(
            targets - pred[None, :],
            axis=1
        )

        normalized_errors.append(
            distances.min() / row.scale
        )

    df["normalized_error"] = normalized_errors
    df["correct"] = df["normalized_error"] <= alpha

    score = df["correct"].mean()

    return score, df


# ============================================================
# 6. RUN BASELINE ON VALIDATION SET
# ============================================================

val_predictions = predict_loader(
    val_loader,
    feature_extractor,
    sampling=SOURCE_SAMPLING
)

pck, val_results = evaluate_pck(
    val_predictions,
    val_df,
    alpha=PCK_THRESHOLD
)

print(f"Validation queries: {len(val_results)}")
print(f"PCK@{PCK_THRESHOLD:.2f}: {pck:.4f}")

display(val_results[
    ["id", "x", "y", "confidence",
     "normalized_error", "correct"]
].head())


# ============================================================
# 7. SAVE VALIDATION PREDICTIONS
# ============================================================

val_predictions.to_csv(
    "validation_predictions.csv",
    index=False
)


In [ ]:

# ============================================================
# TEST PREDICTION AND KAGGLE SUBMISSION
# ============================================================

test_predictions = predict_loader(
    test_loader,
    feature_extractor,
    sampling=SOURCE_SAMPLING
)

# Align predictions with the official submission template
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

pred_lookup = test_predictions.set_index("id")

assert len(test_predictions) == len(sample_submission)
assert set(test_predictions["id"]) == set(sample_submission["id"])

submission = pd.DataFrame({
    "id": sample_submission["id"]
})

submission["prediction"] = [
    json.dumps([
        float(pred_lookup.loc[qid, "x"]),
        float(pred_lookup.loc[qid, "y"])
    ])
    for qid in submission["id"]
]

submission.to_csv(
    "submission_dinov2_baseline.csv",
    index=False
)

print("Submission saved:", len(submission), "queries")
display(submission.head())
